In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
paths = []

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        paths.append(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Multiple Linear Regression
## ML Lab 3
**Yasir Ahmad**  

**22MIA1064**

## Import data

In [ ]:
df = pd.read_csv(paths[0])
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.replace(' ', '_')

In [6]:
df.dtypes

Country                             object
Year                                 int64
Status                              object
Life_expectancy_                   float64
Adult_Mortality                    float64
infant_deaths                        int64
Alcohol                            float64
percentage_expenditure             float64
Hepatitis_B                        float64
Measles_                             int64
_BMI_                              float64
under-five_deaths_                   int64
Polio                              float64
Total_expenditure                  float64
Diphtheria_                        float64
_HIV/AIDS                          float64
GDP                                float64
Population                         float64
_thinness__1-19_years              float64
_thinness_5-9_years                float64
Income_composition_of_resources    float64
Schooling                          float64
dtype: object

In [ ]:
df = df.drop(['Country'],axis=1)

In [ ]:
df['Status'].unique()

In [ ]:
df['Status'] = df['Status'].map({'Developing':1, 'Developed':0})

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df = df[['Year', 'Status','Life_expectancy_', 'Adult_Mortality','Alcohol', 'percentage_expenditure', '_BMI_','GDP', 'Population',]]

In [ ]:
df.shape

In [ ]:
df.describe()

## Raw MLR
raw implementation without feature selection

In [ ]:
X0 = df.drop('Life_expectancy_',axis=1)
y0 = df['Life_expectancy_']

In [ ]:
from sklearn.model_selection import train_test_split
X0_train,X0_test,y0_train,y0_test = train_test_split(X0,y0,test_size=0.3,random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression
Rmodel0 = LinearRegression()

In [ ]:
Rmodel0.fit(X0_train,y0_train)

In [ ]:
y0_pred = Rmodel0.predict(X0_test)

### Metrics of raw model

In [ ]:
from sklearn import metrics

mae = metrics.mean_absolute_error(y0_test, y0_pred)
mse = metrics.mean_squared_error(y0_test, y0_pred)
r2 = metrics.r2_score(y0_test, y0_pred)

print("The raw model performance for testing set")
print("--------------------------------------")
print('MAE is {}'.format(mae))
print('MSE is {}'.format(mse))
print('r2 is {}'.format(r2))

raw_model = {'mae':mae,'mse':mse,'r2':r2}

In [ ]:
import statsmodels.api as sm
# Add a constant to the model (intercept)
X0 = sm.add_constant(X0)

# Fit the model
model0 = sm.OLS(y0, X0).fit()

# Print the summary
print(model0.summary())

## Visualize

In [29]:
import matplotlib.pyplot as plt
import seaborn as sns

#understanding the distribution with seaborn
with sns.plotting_context("notebook",font_scale=2.5):
    g = sns.pairplot(df,hue='Alcohol', palette='tab20',size=6)
g.set(xticklabels=[]);

/opt/conda/lib/python3.10/site-packages/seaborn/axisgrid.py:2095: UserWarning: The `size` parameter has been renamed to `height`; please update your code.
  warnings.warn(msg, UserWarning)
/opt/conda/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/opt/conda/lib/python3.10/site-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/opt/conda/lib/python3.10/site-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` 

Error in callback <function flush_figures at 0x787fed139d80> (for post_execute), with arguments args (),kwargs {}:



KeyboardInterrupt



In [ ]:
cor = df.corr()
sns.heatmap(cor,annot = True)

In [ ]:
cor["Performance_Index"]

## ANOVA

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
for c in df.columns:
    mymod = ols(f'Performance_Index ~ {c}', data = df).fit()  
    # performing type 2 anova test  
    aovtable = sm.stats.anova_lm(mymod, typ = 2)  
    print(f'ANOVA table for Performance_Index ~ {c}')  
    print('------------------------------------------------------------------')  
    print(aovtable)  
    print()

In [ ]:
mymod = ols(f'Performance_Index ~ Hours_Studied+Previous_Scores+Extracurricular_Activities+Sleep_Hours+Sample_Question_Papers_Practiced', data = df).fit()  
# performing type 2 anova test  
aovtable = sm.stats.anova_lm(mymod, typ = 2)  
print(f'ANOVA summary')  
print(aovtable)  
print()

In [ ]:
aovtable['F']

In [ ]:
X = df[['Hours_Studied','Previous_Scores']]
y = df['Performance_Index']

In [ ]:
X

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)

In [ ]:
from sklearn.linear_model import LinearRegression
model2 = LinearRegression()
model2.fit(X_train, y_train)

In [ ]:
# Predicting the Test set results
y_pred = model2.predict(X_test)

In [ ]:
print(y_pred)

## Regressor line

In [ ]:
model2.coef_

In [ ]:
model2.intercept_

In [ ]:
plt.scatter(X_test['Hours_Studied'],y_test)
plt.scatter(X_test['Hours_Studied'],y_pred)
plt.xlabel('Hours_Studied')
plt.ylabel('Performance Index')
plt.show()

In [ ]:
plt.scatter(X_test['Previous_Scores'],y_test)
plt.scatter(X_test['Previous_Scores'],y_pred)
plt.xlabel('Previous_Scores')
plt.ylabel('Performance Index')
plt.show()

In [ ]:
#xr = [-10:11]
x_values = np.arange(0,15)
y_values = model2.coef_[0] * x_values - model2.intercept_
plt.plot(x_values, y_values, label='Regression Line', linestyle='-', marker='o', color='b')
plt.xlabel('Hours Studied')
plt.ylabel('Performance Index')
plt.show()

In [ ]:
#xr = [-10:11]
x_values = np.arange(0,101,5)
y_values = model2.coef_[1] * x_values - model2.intercept_
plt.plot(x_values, y_values, label='Regression Line', linestyle='-', marker='o', color='b')
plt.xlabel('Previous Scores')
plt.ylabel('Performance Index')
plt.show()

In [ ]:
# model evaluation for testing set
from sklearn import metrics

mae = metrics.mean_absolute_error(y_test, y_pred)
mse = metrics.mean_squared_error(y_test, y_pred)
r2 = metrics.r2_score(y_test, y_pred)

print("The model performance for testing set")
print("--------------------------------------")
print('MAE is {}'.format(mae))
print('MSE is {}'.format(mse))
print('r2 is {}'.format(r2))

new_model = {'mae':mae,'mse':mse,'r2':r2}

In [ ]:
import statsmodels.api as sm
# Add a constant to the model (intercept)
X = sm.add_constant(X)

# Fit the model
model = sm.OLS(y, X).fit()

# Print the summary
print(model.summary())

In [ ]:
for i in raw_model:
    print(f"{i}")
    print(f"\traw: {raw_model[i]}")
    print(f"\tnew: {new_model[i]}")
    print(f"\tdiff: {raw_model[i]-new_model[i]}")